# `ptof_obs_nightly_baseline`

## What this notebook does
Recomputes the statistical baseline that detection depends on to tell "normal" from
"anomalous": per-capability, per-field response presence rate. Nothing in this notebook detects
anything itself — it produces the reference point `ptof_obs_mal_output` compares live data against
for schema drift detection.

## Position in the pipeline
- **Separate scheduled job** (`obs_nightly_baseline`) — runs nightly, independently of
  `obs_fresh_scan`. Not one of `obs_fresh_scan`'s tasks.
- **Upstream:** reads `v_llm_bronze` (built by `ptof_obs_bronze_projection`) and
  `capability_registry` (human-curated by `ptof_obs_setup_seed`).
- **Downstream:** `ptof_obs_mal_output` reads `response_field_baseline` to detect schema drift
  (a field that used to reliably appear has gone missing).

## Why baselines are computed nightly, separately from detection
The query does a `CREATE OR REPLACE` over a 30-day rolling window — expensive relative to the
hourly detection queries. Splitting this into its own nightly job means detection runs stay cheap
and compare against a stable reference point.

## Tables/views touched
- **Reads:** `v_llm_bronze`, `capability_registry`
- **Writes:** `response_field_baseline` (per-capability, per-field presence rate)

## Dropped baselines (prod migration 2026-09-10)
- `capability_latency_baseline` — no `latency_ms` in prod. Phase 2 via dev enrichment.

In [ ]:
%sql
-- response_field_baseline: per-capability, per-field presence rate computed nightly over a
-- 30-day rolling window. Supports ptof_obs_mal_output's response_schema_drift detector.
-- Prod context: all rows in ai_shift_outputs are successful outputs (no success/error columns),
-- so no success or credential-fastfail filter is needed. response_parsed is aliased from
-- content in the view, which is 100% valid JSON in prod.
-- Floor of 20 eligible rows per capability: prevents thin data from creating a noisy baseline.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.response_field_baseline AS
WITH eligible AS (
    -- non-blank outputs from active capabilities in the last 30 days
    SELECT b.capability, b.response_parsed
    FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
    JOIN mq_gmdf_dev.oil_obs.capability_registry r
      ON r.capability = b.capability AND r.active = true
    WHERE b.called_at >= current_timestamp() - INTERVAL 30 DAYS
      AND b.is_blank_output = false
),
row_counts AS (
    -- per-capability row counts, filtered to >= 20 to avoid thin baselines
    SELECT capability, count(*) AS n_rows FROM eligible GROUP BY capability HAVING count(*) >= 20
),
field_counts AS (
    -- explode each output's JSON keys and count how many rows each field appears in
    SELECT e.capability, k.key AS field_name, count(*) AS baseline_present
    FROM eligible e
    LATERAL VIEW explode(from_json(cast(e.response_parsed AS STRING), 'map<string,string>')) k AS key, val
    WHERE e.capability IN (SELECT capability FROM row_counts)
    GROUP BY e.capability, k.key
)
SELECT f.capability, f.field_name, f.baseline_present,
       r.n_rows AS baseline_total,
       f.baseline_present * 1.0 / r.n_rows AS baseline_presence_rate,
       current_timestamp() AS computed_at
FROM field_counts f
JOIN row_counts r ON r.capability = f.capability;